# Step 1.1: Parse the metadata.json files for DFDC Parts 0-6

In this step, we will locate the `metadata.json` files for each of the 7 parts (Parts 0 to 6) of the DFDC training dataset, load them, and aggregate their entries into a single unified pandas DataFrame.

In [ ]:
import os
import json
import pandas as pd

# Path to the dataset root folder
dataset_dir = "dfdc_train_dataset"

# Aggregate metadata from parts 0-6
metadata_list = []

for part_idx in range(7):
    part_folder = f"dfdc_train_part_{part_idx}"
    metadata_path = os.path.join(dataset_dir, part_folder, "metadata.json")
    
    if os.path.exists(metadata_path):
        with open(metadata_path, "r") as f:
            part_data = json.load(f)
            
            for video_name, info in part_data.items():
                metadata_list.append({
                    "video_name": video_name,
                    "part": part_idx,
                    "label": info.get("label"),
                    "split": info.get("split"),
                    "original": info.get("original", None),
                    "path": os.path.join(dataset_dir, part_folder, video_name)
                })
    else:
        print(f"Warning: {metadata_path} not found.")

# Convert to a pandas DataFrame
df_metadata = pd.DataFrame(metadata_list)

# Print overview statistics
print(f"Successfully parsed {len(df_metadata)} videos across parts 0-6.")
print("\nLabel distribution:")
print(df_metadata["label"].value_counts())

# Display the first few entries
df_metadata.head()

# Step 1.2: Balance the Dataset

The DFDC dataset is heavily imbalanced towards FAKE videos (e.g. 12,275 FAKEs vs 1,609 REALs in our parsed subset). To ensure our model does not learn a majority-class bias, we will write a balancing script that downsamples the FAKE videos to match the exact number of REAL videos.

In [ ]:
# Separate REAL and FAKE videos
df_real = df_metadata[df_metadata["label"] == "REAL"]
df_fake = df_metadata[df_metadata["label"] == "FAKE"]

# Determine the target size (minimum count of the two)
min_count = min(len(df_real), len(df_fake))

# Sample FAKE videos to match the number of REAL videos
df_fake_balanced = df_fake.sample(n=min_count, random_state=42)

# Concatenate to get the balanced dataset
df_balanced = pd.concat([df_real, df_fake_balanced]).reset_index(drop=True)

# Print balancing results
print(f"Target size per class: {min_count}")
print(f"Total balanced dataset size: {len(df_balanced)}")
print("\nBalanced Label distribution:")
print(df_balanced["label"].value_counts())

# Display a sample of the balanced dataset
df_balanced.sample(5, random_state=42)

# Steps 1.3 - 1.5: Video Ingestion, Facial Detection, and Bounding Box Expansion

Here we develop the complete preprocessing pipeline to extract training face-crops from our balanced dataset:
- **Step 1.3:** OpenCV video ingestion to sample 15 equidistant frames per target video.
- **Step 1.4:** MTCNN for facial detection on the sampled frames. Only process videos where a face is detected (skip otherwise).
- **Step 1.5:** Apply bounding box expansion logic (30% margin) and save face crops as `.jpg` files organized into `train/REAL`, `train/FAKE`, `val/REAL`, and `val/FAKE` directories based on the deterministic Video-ID split rule. 
~280 min

In [3]:
import os
import cv2
import json
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm
from pathlib import Path

# Constants for frame extraction and face detection
OUTPUT_ROOT = "processed_faces_v2"
NUM_FRAMES = 15                  # Number of frames to sample per video
MARGIN = 0.30                    # Margin to expand bounding box by 30%
TRAIN_SPLIT_RATIO = 0.8          # Split ratio for train/validation
MIN_FACE_SIZE = 60               # Minimum size of face in pixels
USE_LARGEST_FACE_ONLY = True     # If multiple faces are detected, keep only the largest

# Setup GPU device if available (CUDA or MPS)
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Note: facenet-pytorch MTCNN/resize path can trigger unsupported adaptive pooling on MPS.
# Workaround: run detection on CPU when running on MPS to avoid runtime error.
mtcnn_device = "cpu" if device == "mps" else device
print(f"Using device: {device} for PyTorch | MTCNN running on: {mtcnn_device}")

# Initialize MTCNN detector
mtcnn = MTCNN(keep_all=True, device=mtcnn_device)

# --- Helper Functions ---

def deterministic_split(video_id, train_ratio=0.8):
    """
    Deterministically split a video into train or val based on its video_id hash.
    Ensures that Actor Leakage is prevented by grouping all frames from the same video.
    """
    h = int(hashlib.md5(video_id.encode("utf-8")).hexdigest()[:8], 16)
    return "train" if (h % 1000) / 1000.0 < train_ratio else "val"

def sample_frame_indices(frame_count, num_samples):
    """
    Equidistantly sample num_samples frame indices from the video.
    """
    if frame_count <= num_samples:
        return list(range(frame_count))
    positions = np.linspace(0, frame_count - 1, num_samples, dtype=int)
    return list(np.unique(positions))

def expand_bbox(box, img_w, img_h, margin):
    """
    Expand the detected bounding box by a given margin percentage.
    """
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    x1n = max(0, int(x1 - margin * w))
    y1n = max(0, int(y1 - margin * h))
    x2n = min(img_w, int(x2 + margin * w))
    y2n = min(img_h, int(y2 + margin * h))
    return x1n, y1n, x2n, y2n

Using device: mps for PyTorch | MTCNN running on: cpu


In [4]:
def process_video_pipeline(video_path, video_filename, label, out_root, mtcnn, num_frames=15, margin=0.30):
    """
    Ingests a video, samples equidistant frames, runs MTCNN facial detection,
    applies bounding box expansion, and saves the cropped face.
    Returns:
        int: Number of face crops saved if successful, 0 if skipped/no faces.
    """
    try:
        # Determine the target output directory based on the split rule and label
        split = deterministic_split(video_filename, TRAIN_SPLIT_RATIO)
        out_dir = os.path.join(out_root, split, label)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return 0
        
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if frame_count == 0:
            cap.release()
            return 0
        
        # Sample 15 frames equidistantly
        indices = sample_frame_indices(frame_count, num_samples=num_frames)
        saved_count = 0
        
        # Extract and process each target frame
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret or frame is None:
                continue
            
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)
            
            # Detect faces
            boxes, probs = mtcnn.detect(pil_img)
            if boxes is None:
                continue
                
            valid_boxes = []
            for box in boxes:
                if box is None:
                    continue
                x1, y1, x2, y2 = box
                width, height = int(x2 - x1), int(y2 - y1)
                # Filter out faces that are too small
                if width < MIN_FACE_SIZE or height < MIN_FACE_SIZE:
                    continue
                valid_boxes.append(box)
                
            if not valid_boxes:
                continue
            
            # Select target face(s)
            if USE_LARGEST_FACE_ONLY and len(valid_boxes) > 1:
                areas = [(b[2]-b[0])*(b[3]-b[1]) for b in valid_boxes]
                idx_largest = int(np.argmax(areas))
                faces_to_use = [valid_boxes[idx_largest]]
            else:
                faces_to_use = valid_boxes
                
            # Crop and save faces
            for fi, box in enumerate(faces_to_use):
                x1, y1, x2, y2 = box
                img_w, img_h = pil_img.width, pil_img.height
                x1n, y1n, x2n, y2n = expand_bbox((x1, y1, x2, y2), img_w, img_h, margin)
                crop = pil_img.crop((x1n, y1n, x2n, y2n))
                
                # Create unique output filename
                base_name = os.path.splitext(video_filename)[0]
                out_name = f"{base_name}_f{idx:06d}"
                if len(faces_to_use) > 1:
                    out_name += f"_p{fi}"
                
                os.makedirs(out_dir, exist_ok=True)
                out_path = os.path.join(out_dir, out_name + ".jpg")
                crop.save(out_path, quality=95)
                saved_count += 1
                
        cap.release()
        return saved_count

    except Exception as e:
        # Safe logging to prevent crashes
        print(f"\n[Pipeline Exception] Failed to process {video_filename}: {e}")
        return 0

# Create the parent directories for output root
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f"Beginning ingestion pipeline for {len(df_balanced)} balanced videos...")
print(f"Saving face crops to: {OUTPUT_ROOT}\n")

processed_count = 0
skipped_count = 0
total_crops_saved = 0

# Progress bar tracking overall video progression with micro process statistics
pbar = tqdm(df_balanced.iterrows(), total=len(df_balanced), desc="Processing Videos")

for idx, row in pbar:
    video_filename = row["video_name"]
    video_path = row["path"]
    label = row["label"]
    
    # Process the video via pipeline
    crops_saved = process_video_pipeline(
        video_path=video_path,
        video_filename=video_filename,
        label=label,
        out_root=OUTPUT_ROOT,
        mtcnn=mtcnn,
        num_frames=NUM_FRAMES,
        margin=MARGIN
    )
    
    if crops_saved > 0:
        processed_count += 1
        total_crops_saved += crops_saved
    else:
        skipped_count += 1
        
    # Real-time micro statistics tracking directly on progression bar
    pbar.set_postfix({
        "Crops Saved": total_crops_saved,
        "Processed": processed_count,
        "Skipped": skipped_count
    })

print(f"\n{'='*50}")
print("Pipeline Processing Finished!")
print(f"Total Videos Processed (with faces found): {processed_count}")
print(f"Total Videos Skipped (no faces or error): {skipped_count}")
print(f"Total Face Crops Saved: {total_crops_saved}")
print(f"{'='*50}")

Beginning ingestion pipeline for 3218 balanced videos...
Saving face crops to: processed_faces_v2



Processing Videos:   0%|          | 0/3218 [00:00<?, ?it/s]


Pipeline Processing Finished!
Total Videos Processed (with faces found): 3218
Total Videos Skipped (no faces or error): 0
Total Face Crops Saved: 47579


# Verification: Count Extracted Face Crops

Once the pipeline completes, we can run this cell to count and verify the split distribution of the extracted face images in each train and validation folder.

In [5]:
import os

# Explicitly define the directory name to prevent NameError if this cell is executed independently
OUTPUT_ROOT = "processed_faces_v2"

# Check counts in output directories
for split in ["train", "val"]:
    for label in ["REAL", "FAKE"]:
        folder = os.path.join(OUTPUT_ROOT, split, label)
        count = 0
        if os.path.isdir(folder):
            count = len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])
        print(f"{split}/{label}: {count} face images")

train/REAL: 18959 face images
train/FAKE: 18953 face images
val/REAL: 4919 face images
val/FAKE: 4748 face images


# Phase 2: Model Setup & Hyperparameter Tuning

Now that our face crop dataset has been extracted and balanced, we will configure the model architecture and data ingestion for training:
- **Step 2.1:** Load the pre-trained Hugging Face ViT model and image processor.
- **Step 2.2:** Define our custom `ViTDeepfakeClassifier` PyTorch module.
- **Step 2.3:** Set up the PyTorch Dataset (`ImageFolder`) and `DataLoader` instances with data augmentations.
- **Step 2.4:** Set up device placement (CUDA/MPS fallback to CPU).
- **Step 2.5:** Set up optimizer and learning rate scheduler.

In [6]:
import torch
import torch.nn as nn
from transformers import ViTImageProcessor, ViTModel

# Step 2.1: Initialize Hugging Face ViT base model and processor
model_name = "google/vit-base-patch16-224-in21k"
print(f"Loading pre-trained processor and base ViT from checkpoint: {model_name}...")
processor = ViTImageProcessor.from_pretrained(model_name)
vit_base = ViTModel.from_pretrained(model_name)
print("Base model loaded successfully.")

# Step 2.2: Construct the custom PyTorch nn.Module class
class ViTDeepfakeClassifier(nn.Module):
    def __init__(self, vit_model, dropout_rate=0.2):
        super().__init__()
        self.vit = vit_model
        self.dropout = nn.Dropout(dropout_rate)
        # Classification head outputting a single binary logit
        self.classifier = nn.Linear(self.vit.config.hidden_size, 1)
        
    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        # Extract the representation of the [CLS] token
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits

# Initialize our classifier wrapper
model = ViTDeepfakeClassifier(vit_base)
print(f"Classifier initialized. Hidden dimension size: {vit_base.config.hidden_size}")

Loading pre-trained processor and base ViT from checkpoint: google/vit-base-patch16-224-in21k...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Base model loaded successfully.
Classifier initialized. Hidden dimension size: 768


In [7]:
import os
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Step 2.3: Configure data transforms and loaders
mean = processor.image_mean
std = processor.image_std
size = processor.size['height'] if 'height' in processor.size else 224
print(f"Configuring image size to {size}x{size} with mean={mean} and std={std}")

# Define image augmentations for training data
train_transforms = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Define standard normalization transforms for validation data
val_transforms = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Set up PyTorch ImageFolder datasets
OUTPUT_ROOT = "processed_faces_v2"
train_dataset = ImageFolder(root=os.path.join(OUTPUT_ROOT, "train"), transform=train_transforms)
val_dataset = ImageFolder(root=os.path.join(OUTPUT_ROOT, "val"), transform=val_transforms)

# Define DataLoader configurations
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

print(f"Data loaders created successfully.")
print(f"Training dataset size: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"Validation dataset size: {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"Class mapping: {train_dataset.class_to_idx}")

Configuring image size to 224x224 with mean=(0.5, 0.5, 0.5) and std=(0.5, 0.5, 0.5)
Data loaders created successfully.
Training dataset size: 37912 images (4739 batches)
Validation dataset size: 9667 images (1209 batches)
Class mapping: {'FAKE': 0, 'REAL': 1}


In [8]:
# Step 2.4: Set up device mapping for hardware acceleration
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(f"Mapping model onto device: {device}")
model = model.to(device)

Mapping model onto device: mps


In [9]:
import torch.optim as optim
from transformers import get_cosine_schedule_with_warmup

# Step 2.5: Configure training parameters, optimizer, and warmup scheduler
EPOCHS = 15
learning_rate = 3e-5
weight_decay = 0.05
ACCUMULATION_STEPS = 4  # Accumulate gradients over 4 batches (effective batch size 32)

# Calculate training steps for scheduler
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_steps)

# Initialize AdamW optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Initialize Cosine Warmup scheduler
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Optimizer & Scheduler configuration:")
print(f"- Optimizer: AdamW (lr={learning_rate}, weight_decay={weight_decay})")
print(f"- Scheduler: Cosine Warmup with Warmup Ratio of 10% (Effective Batch Size 32) (Effective Batch Size 32)")
print(f"- Total Training steps: {total_steps} (Warmup steps: {warmup_steps})")

Optimizer & Scheduler configuration:
- Optimizer: AdamW (lr=3e-05, weight_decay=0.05)
- Scheduler: Cosine Warmup with Warmup Ratio of 10% (Effective Batch Size 32) (Effective Batch Size 32)
- Total Training steps: 17760 (Warmup steps: 1776)


In [10]:
# Verification: Run a test pass with a single batch
print("Extracting a test batch from the training loader...")
images, labels = next(iter(train_loader))
print(f"Batch loaded. Images shape: {images.shape} | Labels shape: {labels.shape}")

# Move data to target device
images_device = images.to(device)
labels_device = labels.to(device).float().unsqueeze(1)

# Set model in training mode and run forward pass
model.train()
with torch.set_grad_enabled(True):
    logits = model(images_device)
    print(f"Forward pass successful. Output logits shape: {logits.shape}")
    
    # Calculate initial loss (using BCEWithLogitsLoss)
    criterion = nn.BCEWithLogitsLoss()
    loss = criterion(logits, labels_device)
    print(f"Initial dummy batch loss: {loss.item():.4f}")

Extracting a test batch from the training loader...


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Batch loaded. Images shape: torch.Size([8, 3, 224, 224]) | Labels shape: torch.Size([8])
Forward pass successful. Output logits shape: torch.Size([8, 1])
Initial dummy batch loss: 0.6951


# Phase 3: Training & Evaluation

Now we will execute the training and evaluation loop. We will train for 5 epochs, monitoring training loss vs validation loss, and computing Accuracy, Precision, Recall, F1-Score, and ROC-AUC. We will save the best model weights dynamically based on validation ROC-AUC improvement. (369 min) | (612 min)

In [11]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from tqdm.notebook import tqdm

# Define criterion
criterion = nn.BCEWithLogitsLoss()

# Tracking metrics for plotting or summaries
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_auc": []
}

best_val_auc = 0.0
best_val_loss = float('inf')
patience = 3
patience_counter = 0

print(f"Starting training on device: {device}")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    processed_samples = 0
    
    # Progress bar for training batches
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    optimizer.zero_grad()
    
    # Sub-progression log interval (every 10%)
    log_interval = max(1, len(train_loader) // 10)
    
    for batch_idx, (images, labels) in enumerate(train_bar):
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        
        logits = model(images)
        loss = criterion(logits, labels)
        
        # Scale the loss to account for gradient accumulation
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        # Perform step only every ACCUMULATION_STEPS batches
        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        running_loss += loss.item() * ACCUMULATION_STEPS * images.size(0)
        processed_samples += images.size(0)
        train_bar.set_postfix(loss=f"{(loss.item() * ACCUMULATION_STEPS):.4f}")
        
        # Print sub-progression status every 10%
        if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
            elapsed_batch_time = time.time() - epoch_start_time
            batches_completed = batch_idx + 1
            it_per_sec = batches_completed / elapsed_batch_time
            batches_remaining = len(train_loader) - batches_completed
            eta_sec = batches_remaining / it_per_sec if it_per_sec > 0 else 0
            current_lr = optimizer.param_groups[0]['lr']
            
            print(f"  -> Progress: {batches_completed}/{len(train_loader)} batches ({int(batches_completed/len(train_loader)*100)}%) | "
                  f"Running Loss: {(running_loss / processed_samples):.4f} | "
                  f"LR: {current_lr:.2e} | "
                  f"ETA: {int(eta_sec // 60)}m {int(eta_sec % 60)}s")
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_probs = []
    all_labels = []
    
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device)
            labels_device = labels.to(device).float().unsqueeze(1)
            
            logits = model(images)
            loss = criterion(logits, labels_device)
            val_loss += loss.item() * images.size(0)
            
            # Sigmoid outputs for evaluation metrics
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            
            all_preds.extend(preds.flatten())
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())
            val_bar.set_postfix(loss=f"{loss.item():.4f}")
            
    epoch_val_loss = val_loss / len(val_dataset)

    # Early stopping implementation
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"Warning: Validation loss did not improve. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print("Early stopping triggered! Training stopped early to protect model weights.")
        break

    
    # Calculate performance metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    
    # Update history
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_accuracy"].append(accuracy)
    history["val_precision"].append(precision)
    history["val_recall"].append(recall)
    history["val_f1"].append(f1)
    history["val_auc"].append(auc)
    
    # Timing and estimation
    elapsed_time = time.time() - start_time
    epoch_duration = time.time() - epoch_start_time
    remaining_epochs = EPOCHS - epoch
    estimated_remaining_time = epoch_duration * remaining_epochs
    
    print(f"\n--- Epoch {epoch} Metrics Summary ---")
    print(f"  Train Loss:       {epoch_train_loss:.4f}")
    print(f"  Val Loss:         {epoch_val_loss:.4f}")
    print(f"  Val Accuracy:     {accuracy:.4f}")
    print(f"  Val Precision:    {precision:.4f}")
    print(f"  Val Recall:       {recall:.4f}")
    print(f"  Val F1-Score:     {f1:.4f}")
    print(f"  Val ROC-AUC:      {auc:.4f}")
    print(f"  Epoch Duration:   {int(epoch_duration // 60)}m {int(epoch_duration % 60)}s")
    print(f"  Elapsed Time:     {int(elapsed_time // 60)}m {int(elapsed_time % 60)}s")
    if remaining_epochs > 0:
        print(f"  Estimated Remaining Time: {int(estimated_remaining_time // 60)}m {int(estimated_remaining_time % 60)}s")
    else:
        print("  Training Completed!")
        
    # --- BEST MODEL CHECKPOINTING ---
    if auc > best_val_auc:
        print(f"  => Validation ROC-AUC improved from {best_val_auc:.4f} to {auc:.4f}. Saving best model weight checkpoint to 'best_vit_deepfake_2.pth'...")
        best_val_auc = auc
        torch.save(model.state_dict(), "best_vit_deepfake_2.pth")
    print("-" * 40)


Starting training on device: mps


Epoch 1/15 [Train]:   0%|          | 0/4739 [00:00<?, ?it/s]

  -> Progress: 473/4739 batches (9%) | Running Loss: 0.6947 | LR: 1.99e-06 | ETA: 52m 5s
  -> Progress: 946/4739 batches (19%) | Running Loss: 0.6856 | LR: 3.99e-06 | ETA: 50m 45s
  -> Progress: 1419/4739 batches (29%) | Running Loss: 0.6655 | LR: 5.98e-06 | ETA: 45m 54s
  -> Progress: 1892/4739 batches (39%) | Running Loss: 0.6392 | LR: 7.99e-06 | ETA: 40m 55s
  -> Progress: 2365/4739 batches (49%) | Running Loss: 0.6031 | LR: 9.98e-06 | ETA: 34m 19s
  -> Progress: 2838/4739 batches (59%) | Running Loss: 0.5673 | LR: 1.20e-05 | ETA: 27m 43s
  -> Progress: 3311/4739 batches (69%) | Running Loss: 0.5345 | LR: 1.40e-05 | ETA: 21m 1s
  -> Progress: 3784/4739 batches (79%) | Running Loss: 0.5044 | LR: 1.60e-05 | ETA: 14m 12s
  -> Progress: 4257/4739 batches (89%) | Running Loss: 0.4797 | LR: 1.80e-05 | ETA: 7m 13s
  -> Progress: 4730/4739 batches (99%) | Running Loss: 0.4599 | LR: 2.00e-05 | ETA: 0m 8s
  -> Progress: 4739/4739 batches (100%) | Running Loss: 0.4596 | LR: 2.00e-05 | ETA: 0m 

Epoch 1/15 [Val]:   0%|          | 0/1209 [00:00<?, ?it/s]


--- Epoch 1 Metrics Summary ---
  Train Loss:       0.4596
  Val Loss:         0.3008
  Val Accuracy:     0.8802
  Val Precision:    0.9248
  Val Recall:       0.8323
  Val F1-Score:     0.8761
  Val ROC-AUC:      0.9509
  Epoch Duration:   77m 35s
  Elapsed Time:     77m 35s
  Estimated Remaining Time: 1086m 19s
  => Validation ROC-AUC improved from 0.0000 to 0.9509. Saving best model weight checkpoint to 'best_vit_deepfake_2.pth'...
----------------------------------------


...

Epoch 7/15 [Train]:   0%|          | 0/4739 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  -> Progress: 473/4739 batches (9%) | Running Loss: 0.0698 | LR: 2.22e-05 | ETA: 63m 12s
  -> Progress: 946/4739 batches (19%) | Running Loss: 0.0687 | LR: 2.19e-05 | ETA: 56m 9s
  -> Progress: 1419/4739 batches (29%) | Running Loss: 0.0712 | LR: 2.16e-05 | ETA: 49m 10s
  -> Progress: 1892/4739 batches (39%) | Running Loss: 0.0736 | LR: 2.12e-05 | ETA: 42m 12s
  -> Progress: 2365/4739 batches (49%) | Running Loss: 0.0723 | LR: 2.09e-05 | ETA: 35m 14s
  -> Progress: 2838/4739 batches (59%) | Running Loss: 0.0706 | LR: 2.06e-05 | ETA: 28m 15s
  -> Progress: 3311/4739 batches (69%) | Running Loss: 0.0707 | LR: 2.03e-05 | ETA: 21m 15s
  -> Progress: 3784/4739 batches (79%) | Running Loss: 0.0697 | LR: 2.00e-05 | ETA: 14m 14s
  -> Progress: 4257/4739 batches (89%) | Running Loss: 0.0689 | LR: 1.96e-05 | ETA: 7m 11s
  -> Progress: 4730/4739 batches (99%) | Running Loss: 0.0678 | LR: 1.93e-05 | ETA: 0m 8s
  -> Progress: 4739/4739 batches (100%) | Running Loss: 0.0677 | LR: 1.93e-05 | ETA: 0m

Epoch 7/15 [Val]:   0%|          | 0/1209 [00:00<?, ?it/s]

Early stopping triggered! Training stopped early to protect model weights.


In [12]:
# ============================================================
# Model Comparison: best_vit_deepfake.pth vs best_vit_deepfake_2.pth
# ============================================================
import os
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from tqdm.notebook import tqdm

checkpoints = {
    'Model v1 (best_vit_deepfake.pth)': 'best_vit_deepfake.pth',
    'Model v2 (best_vit_deepfake_2.pth)': 'best_vit_deepfake_2.pth'
}

results = {}

for name, path in checkpoints.items():
    if not os.path.exists(path):
        print(f"⚠️  {path} not found, skipping.")
        continue
        
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")
    
    # Load weights into a fresh model copy
    eval_model = ViTDeepfakeClassifier(vit_base, dropout_rate=0.2)
    eval_model.load_state_dict(torch.load(path, map_location=device))
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    # File size
    file_size_mb = os.path.getsize(path) / (1024 * 1024)
    
    # Evaluate on validation set
    all_preds = []
    all_probs = []
    all_labels = []
    total_val_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validating {path}"):
            images = images.to(device)
            labels_device = labels.to(device).float().unsqueeze(1)
            
            logits = eval_model(images)
            loss = criterion(logits, labels_device)
            total_val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            
            all_preds.extend(preds.flatten())
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())
    
    # Calculate metrics
    val_loss = total_val_loss / len(val_dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    
    results[name] = {
        'val_loss': val_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'file_size_mb': file_size_mb,
        'confusion_matrix': cm
    }
    
    print(f"  File Size:       {file_size_mb:.2f} MB")
    print(f"  Val Loss:        {val_loss:.4f}")
    print(f"  Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision:       {precision:.4f} ({precision*100:.2f}%)")
    print(f"  Recall:          {recall:.4f} ({recall*100:.2f}%)")
    print(f"  F1-Score:        {f1:.4f} ({f1*100:.2f}%)")
    print(f"  ROC-AUC:         {auc:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    TN={cm[0][0]}  FP={cm[0][1]}")
    print(f"    FN={cm[1][0]}  TP={cm[1][1]}")

# ---- Side-by-Side Comparison ----
if len(results) == 2:
    names = list(results.keys())
    r1, r2 = results[names[0]], results[names[1]]
    
    print(f"\n\n{'='*60}")
    print(f"  SIDE-BY-SIDE COMPARISON")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'v1':>12} {'v2':>12} {'Delta':>12}")
    print(f"{'-'*56}")
    
    for metric, label in [('val_loss', 'Val Loss'), ('accuracy', 'Accuracy'),
                           ('precision', 'Precision'), ('recall', 'Recall'),
                           ('f1', 'F1-Score'), ('auc', 'ROC-AUC')]:
        v1_val = r1[metric]
        v2_val = r2[metric]
        delta = v2_val - v1_val
        arrow = '▲' if delta > 0 else '▼' if delta < 0 else '='
        # For val_loss, lower is better, so flip the arrow meaning
        if metric == 'val_loss':
            status = '✅' if delta < 0 else '❌'
        else:
            status = '✅' if delta > 0 else '❌' if delta < 0 else '—'
        print(f"{label:<20} {v1_val:>12.4f} {v2_val:>12.4f} {arrow}{abs(delta):>10.4f} {status}")
    
    print(f"{'-'*56}")
    print(f"{'File Size (MB)':<20} {r1['file_size_mb']:>12.2f} {r2['file_size_mb']:>12.2f}")
    print(f"{'='*60}")
    
    # Determine winner
    v2_wins = 0
    if r2['val_loss'] < r1['val_loss']: v2_wins += 1
    if r2['accuracy'] > r1['accuracy']: v2_wins += 1
    if r2['f1'] > r1['f1']: v2_wins += 1
    if r2['auc'] > r1['auc']: v2_wins += 1
    
    if v2_wins >= 3:
        print("\n🏆 Verdict: Model v2 wins on majority of metrics!")
    elif v2_wins <= 1:
        print("\n🏆 Verdict: Model v1 still leads on majority of metrics.")
    else:
        print("\n⚖️  Verdict: Mixed results — v2 has better generalization (lower val loss), v1 has slightly higher raw scores.")
    
    print("\nNote: v1 metrics may be inflated due to overfitting (train-val gap was 0.1844 vs v2's 0.1066).")
    print("v2 is expected to perform better on truly unseen data.")



Evaluating: Model v1 (best_vit_deepfake.pth)


Validating best_vit_deepfake.pth:   0%|          | 0/1209 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  File Size:       329.62 MB
  Val Loss:        0.2440
  Accuracy:        0.9233 (92.33%)
  Precision:       0.9230 (92.30%)
  Recall:          0.9266 (92.66%)
  F1-Score:        0.9248 (92.48%)
  ROC-AUC:         0.9757
  Confusion Matrix:
    TN=4368  FP=380
    FN=361  TP=4558

Evaluating: Model v2 (best_vit_deepfake_2.pth)


Validating best_vit_deepfake_2.pth:   0%|          | 0/1209 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  File Size:       329.62 MB
  Val Loss:        0.2199
  Accuracy:        0.9139 (91.39%)
  Precision:       0.9144 (91.44%)
  Recall:          0.9166 (91.66%)
  F1-Score:        0.9155 (91.55%)
  ROC-AUC:         0.9738
  Confusion Matrix:
    TN=4326  FP=422
    FN=410  TP=4509


  SIDE-BY-SIDE COMPARISON
Metric                         v1           v2        Delta
--------------------------------------------------------
Val Loss                   0.2440       0.2199 ▼    0.0241 ✅
Accuracy                   0.9233       0.9139 ▼    0.0094 ❌
Precision                  0.9230       0.9144 ▼    0.0086 ❌
Recall                     0.9266       0.9166 ▼    0.0100 ❌
F1-Score                   0.9248       0.9155 ▼    0.0093 ❌
ROC-AUC                    0.9757       0.9738 ▼    0.0020 ❌
--------------------------------------------------------
File Size (MB)             329.62       329.62

🏆 Verdict: Model v1 still leads on majority of metrics.

Note: v1 metrics may be inflated due to overfi